# Relevia research: Colab execution scaffold

I want experiment definitions to remain inspectable outside a notebook, so the current scaffold calls scripts in `src/` and configurations in `configs/`. That separation helps review; it does not by itself guarantee reproducibility. Exact revisions, data hashes, runtime versions and effective settings still need to be recorded.

This notebook has no recorded model results. It is retained for later investigation, not for execution during Milestone 2. The revision-pinned Experiment 01 harness is the current entry point for controlled preparation. Hardware capacity and runtime must be measured rather than inferred from a GPU name.

Milestone 2: prompt selection uses dev only. Few-shot conversations come from reserved official training groups. Formerly exposed validation groups are quarantined. Run scripts/check_leakage.py before generation. These legacy notebooks do not substitute for the revision-pinned Experiment 01 harness. No training is authorized in Milestone 2.


## 1. Get the project
Upload the `relevia-research` folder to Google Drive (under `MyDrive/`), then mount it. Adapters and results are saved straight to Drive, to reduce the risk of losing completed artifacts after disconnection; interrupted or incomplete writes still need checking.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/relevia-research

In [ ]:
!pip install -q -r requirements.txt

## 2. Log in to Hugging Face
Needed for the Llama models (accept Meta's licence on the model page first). Qwen doesn't need it.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Build and inspect the data

Related turns must remain within one conversation partition; removing only a few-shot row does not remove the context shared with later turns. The builder now reserves development groups from training and quarantines the formerly exposed validation groups. Inspect the split manifest and run the leakage checker before generating responses. Counts alone cannot rule out duplicated text or pretraining contamination.

In [ ]:
!python data/build_dataset.py
!cat data/processed/split_info.json

## 4. Select the proposed model

The configuration selects the model family and the seed, but the legacy runner does not meet the new exact-revision logging contract. Use the Experiment 01 harness for recorded baseline work. Keep the model, tokenizer and decoding settings fixed when testing a prompt change; otherwise its effect cannot be isolated.

In [ ]:
MODEL = 'configs/models/llama-3.2-3b.yaml'   # or llama-3.2-1b / qwen2.5-3b / qwen2.5-1.5b
SEED = 1

## 5. Deferred smoke checks

A small generation batch can expose loading and formatting failures before a larger run. The training command below is also a deferred operation and must not be executed during Milestone 2. A successful smoke test would establish that the selected path runs, not that its responses are useful or safe. Runtime is not yet measured.

In [ ]:
!python src/generate.py --model {MODEL} --condition zero_shot --testset dev --limit 16
!python src/train.py --model {MODEL} --seed 99 --limit 200

## 6. Deferred model comparisons

The loops below belong to the older study scaffold. They include final-test generation, so they must not be used to develop prompts. They also compare adaptation under a plain prompt; the primary B/C comparison now requires the same developed prompt in both conditions. Reconcile this scaffold with the frozen protocol before later execution rather than reporting its conditions as interchangeable.

In [ ]:
# baselines
for cond in ['zero_shot', 'tuned_prompt']:
    for ts in ['test_seen', 'test_unseen', 'safety/crisis', 'safety/sycophancy']:
        !python src/generate.py --model {MODEL} --condition {cond} --testset {ts}

In [ ]:
# fine-tune + generate (repeat with SEED = 2 and 3)
!python src/train.py --model {MODEL} --seed {SEED}
for ts in ['test_seen', 'test_unseen', 'safety/crisis', 'safety/sycophancy']:
    !python src/generate.py --model {MODEL} --condition finetuned --testset {ts} --seed {SEED}

## 7. Interpret the available metrics

The legacy evaluator reports lexical and length-based summaries. These can expose repetitive openings or output changes, but referral words are not evidence that a response handles risk appropriately. Keep these diagnostics separate from blinded quality ratings and independently reviewed safety outcomes. No output is recorded here.

In [ ]:
!python src/evaluate.py
import pandas as pd
pd.read_csv('results/metrics/auto_metrics.csv')

## What remains unresolved
No executed outputs in this notebook establish an improvement in dialogue quality, safety or deployment performance. The next useful evidence is a controlled baseline run after the Milestone 2 gates are resolved. Preserve unsuccessful runs and interpret any later results against the recorded controls, rather than treating a completed cell as a successful experiment.
